# Evaluate Rossmann Subsampled Dataset

This notebook demonstrates how to evaluate the `rossmann_subsampled` dataset using the Syntherela benchmarking framework.
It is designed to run in Google Colab or a similar environment.

## Setup

Install the `syntherela` package and `gdown` for downloading data.

In [4]:
!pip install syntherela gdown

## Download Data

Download and extract the original and synthetic datasets.

In [5]:
import os
import zipfile
import gdown

def download_and_extract(url, filename, extract_path):
    # Create directory
    os.makedirs(extract_path, exist_ok=True)
    
    # Download
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        gdown.download(url, filename, quiet=False)
    
    # Extract
    print(f"Extracting {filename} to {extract_path}...")
    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done.")

# URLs for the datasets
orig_url = "https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG"
synth_url = "https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl"

# Download and extract to ./data/original and ./data/synthetic
download_and_extract(orig_url, "original.zip", os.path.join("data", "original"))
download_and_extract(synth_url, "synthetic.zip", os.path.join("data", "synthetic"))

Downloading...
From (original): https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG
From (redirected): https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG&confirm=t&uuid=06a27ba4-f447-46af-aa34-eb509d03bb22
To: /Users/martinjurkovic/Documents/github_projects/syntherela/examples/original.zip
100%|██████████| 101M/101M [00:04<00:00, 25.3MB/s]


Extracting original.zip to data/original...
Done.


Downloading...
From (original): https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl
From (redirected): https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl&confirm=t&uuid=c3b1ae8e-8a81-443c-9e1b-2cf5914a9e8f
To: /Users/martinjurkovic/Documents/github_projects/syntherela/examples/synthetic.zip
100%|██████████| 960M/960M [00:34<00:00, 28.2MB/s]


Extracting synthetic.zip to data/synthetic...
Done.


## Imports

Import necessary libraries for evaluation.

In [6]:
import logging
import sys
from pathlib import Path
import json

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from syntherela.benchmark import Benchmark
from syntherela.metrics.single_column.distance import (
    HellingerDistance,
    JensenShannonDistance,
    WassersteinDistance,
    TotalVariationDistance,
)
from syntherela.metrics.single_column.statistical import (
    ChiSquareTest,
    KolmogorovSmirnovTest,
)
from syntherela.metrics.single_table.distance import (
    MaximumMeanDiscrepancy,
    PairwiseCorrelationDifference,
)
from syntherela.metrics.single_column.detection import SingleColumnDetection
from syntherela.metrics.single_table.detection import SingleTableDetection
from syntherela.metrics.multi_table.detection import AggregationDetection
from syntherela.metrics.multi_table.statistical import CardinalityShapeSimilarity
from syntherela.utils import NpEncoder

 20%|██        | 193M/960M [02:39<10:32, 1.21MB/s]


## Configuration

Set up the dataset, method, and paths.

In [12]:
dataset_name = "rossmann_subsampled"
method = "SDV"  # You can change this to other methods like RCTGAN, MOSTLYAI, etc.
run_id = "1"

# Paths relative to the current working directory
real_data_dir = os.path.join("data", "original")
synthetic_data_dir = os.path.join("data", "synthetic")
results_dir = "results"

# Ensure results directory exists
os.makedirs(results_dir, exist_ok=True)

## Define Metrics

Initialize the metrics to be used for evaluation.

In [ ]:
xgb_cls = XGBClassifier
xgb_args = {"seed": 0, "verbosity": 0, "use_label_encoder": False, "eval_metric": "logloss"}

single_column_metrics = [
    ChiSquareTest(),
    # KolmogorovSmirnovTest(),
    # TotalVariationDistance(),
    # HellingerDistance(),
    # JensenShannonDistance(),
    # WassersteinDistance(),
    # SingleColumnDetection(
    #     classifier_cls=xgb_cls, classifier_args=xgb_args, random_state=42
    # ),
]

single_table_metrics = [
    MaximumMeanDiscrepancy(),
    # PairwiseCorrelationDifference(),
    # SingleTableDetection(
    #     classifier_cls=xgb_cls, classifier_args=xgb_args, random_state=42
    # ),
]

multi_table_metrics = [
    CardinalityShapeSimilarity(),
    # AggregationDetection(
    #     classifier_cls=xgb_cls, classifier_args=xgb_args, random_state=42
    # ),
]

## Run Benchmark

Initialize the `Benchmark` class and run the evaluation.

In [24]:
benchmark = Benchmark(
    real_data_dir=real_data_dir,
    synthetic_data_dir=synthetic_data_dir,
    results_dir=results_dir,
    benchmark_name="ExampleBenchmark",
    single_column_metrics=single_column_metrics,
    single_table_metrics=single_table_metrics,
    multi_table_metrics=multi_table_metrics,
    run_id=run_id,
    sample_id="sample1",
    datasets=[dataset_name],
    methods=[method],
)

# Run the benchmark
benchmark.run()

Starting benchmark for rossmann_subsampled, method_name SDV


Running Single Column Metrics: 100%|██████████| 19/19 [00:00<00:00, 3088.95it/s]


Generating report ...

(1/1) Evaluating Column Shapes: |██████████| 19/19 [00:00<00:00, 254.54it/s]|
Column Shapes Score: 81.25%

Overall Score (Average): 81.25%



Running Single Table Metrics: 100%|██████████| 2/2 [00:20<00:00, 10.32s/it]


Generating report ...

(1/1) Evaluating Column Pair Trends: |██████████| 81/81 [00:00<00:00, 609.10it/s]|
Column Pair Trends Score: 68.08%



...

/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real = real.groupby(list(columns), dropna=False).size() / len(real)
/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:88: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_synthetic = synthetic.groupby(list(columns), dropna=False).size() / len(
/Users/martinjurkovic/Documents/github_pr

(2/2) Evaluating Intertable Trends: |████████  | 72/90 [00:00<00:00, 366.15it/s]|

/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real = real.groupby(list(columns), dropna=False).size() / len(real)
/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:88: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_synthetic = synthetic.groupby(list(columns), dropna=False).size() / len(
/Users/martinjurkovic/Documents/github_pr

(2/2) Evaluating Intertable Trends: |██████████| 90/90 [00:00<00:00, 344.74it/s]|
Intertable Trends Score: 74.34%

Overall Score (Average): 86.77%

Long Range Scores: {1: np.float64(0.7433871229292182)}
All avg scores:  0.7433871229292182


/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real = real.groupby(list(columns), dropna=False).size() / len(real)
/Users/martinjurkovic/Documents/github_projects/syntherela/.venv/lib/python3.12/site-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:88: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_synthetic = synthetic.groupby(list(columns), dropna=False).size() / len(
/Users/martinjurkovic/Documents/github_pr

## View Results

The results are stored in `benchmark.all_results`. Here we display a summary.

In [25]:
results = benchmark.all_results.get(dataset_name, {}).get(method, {})
with open(os.path.join(results_dir, f"{dataset_name}_{method}_{run_id}_sample1.json"), "r") as f:
    results = json.load(f)

# Print a few key metrics
print(f"Results for {dataset_name} using {method}:")

if "single_column_metrics" in results:
    print("\nSingle Column Metrics (Chi-Square Test):")
    print(json.dumps(results["single_column_metrics"].get("ChiSquareTest", {}), indent=2, cls=NpEncoder))


Results for rossmann_subsampled using SDV:

Single Column Metrics (Chi-Square Test):
{
  "historical": {
    "DayOfWeek": {
      "p_value": 0.0,
      "statistic": 9947.632976704308
    },
    "Open": {
      "p_value": 0.0,
      "statistic": 10790.037302746003
    },
    "Promo": {
      "p_value": 1.4468271126857989e-64,
      "statistic": 287.87116202468746
    },
    "SchoolHoliday": {
      "p_value": 0.0,
      "statistic": 15579.665964401729
    },
    "StateHoliday": {
      "p_value": 1.0,
      "statistic": 0.0
    }
  },
  "store": {
    "Assortment": {
      "p_value": 2.1254423060600378e-72,
      "statistic": 330.06429354165124
    },
    "Promo2": {
      "p_value": 0.13762217723037048,
      "statistic": 2.204346801629361
    },
    "PromoInterval": {
      "p_value": 6.459926417165668e-07,
      "statistic": 28.504955447520626
    },
    "StoreType": {
      "p_value": 7.679144325003361e-97,
      "statistic": 448.28277996055715
    }
  }
}


In [26]:
if "single_table_metrics" in results:
    print("\nSingle Table Metrics (Maximum Mean Discrepancy):")
    print(json.dumps(results["single_table_metrics"].get("MaximumMeanDiscrepancy", {}), indent=2, cls=NpEncoder))


Single Table Metrics (Maximum Mean Discrepancy):
{
  "historical": {
    "bootstrap_mean": 1.781512424829418,
    "bootstrap_se": 0.0002764302987081426,
    "reference_ci": [
      0,
      0.01974407448868399
    ],
    "reference_mean": 7.124527525661668e-05,
    "reference_variance": 7.151288965281656e-05,
    "value": 1.7814127858798916
  },
  "store": {
    "bootstrap_mean": 0.01761974308015158,
    "bootstrap_se": 0.0002858398587593915,
    "reference_ci": [
      0,
      0.19432861529272966
    ],
    "reference_mean": 0.010651001547830248,
    "reference_variance": 0.006233947505965626,
    "value": 0.006899605086857402
  }
}


In [27]:
if "multi_table_metrics" in results:
    print("\nMulti Table Metrics:")
    print(json.dumps(results["multi_table_metrics"], indent=2, cls=NpEncoder))


Multi Table Metrics:
{
  "CardinalityShapeSimilarity": {
    "store_historical": {
      "pval": 0.9999999999999802,
      "statistic": 0.008071748878923767
    }
  },
  "Trends": {
    "cardinality": 0.9919282511210762,
    "k_hop_similarity": {
      "1": {
        "mean": 0.7433871229292182,
        "se": 0.023547260263957193
      }
    }
  }
}
